# Retail Credit Scoring with Logistic Regression

### Credit Scoring Risk Assessment — Banking-AI

This notebook explains each step in plain language so new learners can follow:

## Step 0 — Notebook Header

**Prerequisites:** Basic Python (`pandas`, `numpy`, `matplotlib`), familiarity with `scikit-learn` basics, and basic understanding of credit scoring and risk assessment terminology.

**What You Will Learn:**

After completing this notebook, you will be able to:
- Explain the credit scoring and risk assessment problem and why the chosen classification approach suits this banking workflow.
- Implement an end-to-end classification pipeline on synthetic data and evaluate performance with domain-appropriate metrics.
- Assess whether the model is operationally viable, considering error costs, fairness, and the limitations of synthetic data.

**Estimated time:** 35–45 minutes

**Collection context:** Credit scoring, probability of default, loss estimation, and stress testing for banking risk management.

## Step 1 — Banking Problem Introduction

**Why this notebook matters:** Credit scoring is the most common use of AI in banking. It helps banks decide who gets a loan and at what interest rate. Logistic Regression is the industry standard because it's transparent and regulatory-friendly.

**Input data used:** Annual income, total debt, age, employment length, and previous late payments.

**What we predict:** Binary credit outcome (`good_credit` vs `bad_credit`).

**ML method used:** Logistic Regression.

**Learning goal:** Learn how to build a simple, interpretable credit scorecard.

**Primary stakeholders:** credit analysts, loan officers, risk managers, and regulatory auditors.

**Comprehension check:** Before looking at the data, write one sentence describing the core decision this model supports and who benefits from getting it right.

**Domain framing note:** This notebook uses synthetic data for educational purposes. It demonstrates decision-support prototyping, not production-ready deployment.

## Step 2 — Environment Setup

Import libraries and set deterministic seeds for reproducibility.

In [ ]:
# Requires: numpy>=1.24, pandas>=2.0, scikit-learn>=1.3, matplotlib>=3.7, seaborn>=0.13

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score

sns.set_theme(style="whitegrid")

RANDOM_SEED = 42
RNG = np.random.default_rng(RANDOM_SEED)
plt.rcParams["figure.figsize"] = (10, 6)

print("Environment ready.")


## Step 3 — Data Creation & Context

Build Synthetic Credit Dataset

We generate 5,000 loan applicants with realistic financial attributes.

In [ ]:
n = 5000
rng = RNG

income = rng.normal(50000, 15000, n).clip(15000, 150000)
debt = rng.normal(15000, 10000, n).clip(0, 100000)
age = rng.integers(21, 75, n)
emp_length_yrs = rng.integers(0, 40, n)
late_payments_last_2yr = rng.poisson(0.3, n)

# Logic for Credit Outcome (Probability of Default)
logit_score = (
    -0.5 * (income / 10000) + 
    0.8 * (debt / 10000) + 
    1.5 * late_payments_last_2yr - 
    0.2 * emp_length_yrs +
    rng.normal(0, 1, n)
)

prob_default = 1 / (1 + np.exp(-logit_score))
is_bad_credit = (prob_default > 0.5).astype(int)

df = pd.DataFrame({
    'income': income,
    'debt': debt,
    'age': age,
    'emp_length_yrs': emp_length_yrs,
    'late_payments': late_payments_last_2yr,
    'is_bad_credit': is_bad_credit
})

print(f"Dataset generated. Bad credit cases: {df['is_bad_credit'].sum()} ({df['is_bad_credit'].mean()*100:.1f}%)")

## Step 4 — Exploratory Data Analysis

EDA

Income and late payments are usually the strongest predictors.

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
sns.violinplot(x='is_bad_credit', y='income', data=df)
plt.title('Income vs Credit Outcome')

plt.subplot(1, 2, 2)
sns.countplot(x='late_payments', hue='is_bad_credit', data=df)
plt.title('Late Payments vs Credit Outcome')

plt.tight_layout()
plt.show()
plt.close()

**Alt-text:** Distribution plot, count plot titled "Income vs Credit Outcome" and "Late Payments vs Credit Outcome". The chart highlights distributional differences between groups that inform the modelling approach.

## Step 5 — Feature Engineering

Prepare features for modelling. Domain-meaningful transformations help the model learn patterns aligned with banking practice.

In [ ]:
X = df.drop('is_bad_credit', axis=1)
y = df['is_bad_credit']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_SEED)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## Step 6 — Baseline Establishment

A baseline sets the floor that any useful model must beat. Without it, performance numbers have no context.

In [ ]:
# --- Baseline: always predict the majority class ---
baseline_clf = DummyClassifier(strategy='most_frequent', random_state=RANDOM_SEED)
baseline_clf.fit(X_train_scaled, y_train)
baseline_pred = baseline_clf.predict(X_test_scaled)
baseline_acc = accuracy_score(y_test, baseline_pred)
print(f"Baseline accuracy (majority-class): {baseline_acc:.3f}")
print(f"Any useful model must beat this {baseline_acc:.3f} floor.")

## Step 7 — Model Training & Evaluation

Train the model and compare performance against the baseline.

**Prediction prompt:** Before viewing the results, what performance do you expect and why?

In [ ]:
model = LogisticRegression()
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)
y_proba = model.predict_proba(X_test_scaled)[:, 1]

In [ ]:
# Simple transformation: Score = 850 - (Probability * 550)
credit_scores = 850 - (y_proba * 550)

plt.hist(credit_scores, bins=30, color='#264653', edgecolor='#264653')
plt.title('Distribution of Generated Credit Scores')
plt.xlabel('Credit Score (FICO-like)')
plt.ylabel('Number of Applicants')
plt.tight_layout()
plt.show()
plt.close()

print(f"Average Score for 'Good' Credit: {credit_scores[y_test==0].mean():.0f}")
print(f"Average Score for 'Bad' Credit: {credit_scores[y_test==1].mean():.0f}")

**Alt-text:** Histogram titled "Distribution of Generated Credit Scores" with Credit Score (FICO-like) on the x-axis and Number of Applicants on the y-axis. The chart highlights distributional differences between groups that inform the modelling approach.

## Step 8 — Interpretability & Explainability

Interpret the model in domain terms. A banking professional should be able to assess whether the model's reasoning is credible.

In [ ]:
coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Weight': model.coef_[0]
}).sort_values(by='Weight')

sns.barplot(x='Weight', y='Feature', data=coef_df)
plt.title('Feature Weights: Positive Weight = Higher Risk of Default')
plt.tight_layout()
plt.show()
plt.close()

**Alt-text:** Bar chart titled "Feature Weights: Positive Weight = Higher Risk of Default". The chart ranks features or categories by magnitude to highlight dominant factors.

Let's look at the coefficients (weights) of our model.

## Step 9 — Operational Application

Demonstrate how the model output feeds into a real banking workflow decision.

In [ ]:
# --- Scenario: score representative cases from the test set ---
print("Operational demonstration — model decision support:\n")
proba = model.predict_proba(X_test)[:, 1]
low_idx  = int(np.argmin(proba))
high_idx = int(np.argmax(proba))
mid_idx  = int(np.argsort(proba)[len(proba) // 2])

for label, idx in [("Low-risk", low_idx), ("Medium-risk", mid_idx), ("High-risk", high_idx)]:
    row = X_test.iloc[idx]
    prob = proba[idx]
    print(f"{label} case  predicted probability: {prob:.1%}")
    print(f"  Features: {dict(row.round(2))}")
    print()

print("A decision-maker would combine these scores with business rules and domain judgement.")

## Step 10 — Summary & Reflection

**What you accomplished:** You built an end-to-end credit scoring and risk assessment workflow — from problem framing through data creation, baseline comparison, model training, evaluation, and operational demonstration.

**Limitations:**

- The dataset is synthetic and cannot capture the full complexity of real banking data.
- Model performance on synthetic data does not guarantee equivalent performance in production.
- This notebook is an educational prototype. Real deployment requires validated data, regulatory review, and ongoing monitoring.

**Ethical reflection:**

- Credit models can perpetuate historical discrimination if trained on biased lending data.
- Proxy variables such as zip code or employment type may correlate with protected characteristics.
- Applicants deserve transparent explanations of adverse credit decisions under fair lending regulations.

**Reflection questions:**

1. What additional data sources or features would you need before trusting this model in a live banking environment?
2. How would the relative cost of false positives versus false negatives change the metric you optimise for?
3. How could you adapt this workflow to a related problem in credit scoring and risk assessment?

## References

- Scikit-learn documentation: [https://scikit-learn.org/stable/](https://scikit-learn.org/stable/)
- Project quality baseline: `QUALITY_STANDARD.md`
- Domain context drawn from public banking and financial services literature.